# 07. 스트리밍 — 실시간으로 에이전트 실행 관찰

## 학습 목표

LangGraph의 다양한 스트리밍 모드를 이해하고 활용합니다.

- `values`, `updates`, `messages`, `custom`, `debug` 모드의 차이를 이해합니다
- 각 스트리밍 모드의 적절한 사용 사례를 파악합니다
- 여러 스트리밍 모드를 동시에 사용하는 방법을 익힙니다

## 7.1 환경 설정

In [1]:
from dotenv import load_dotenv
load_dotenv()
from langchain_openai import ChatOpenAI
from langchain.tools import tool
from langchain.messages import HumanMessage, SystemMessage, ToolMessage, AnyMessage
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import InMemorySaver
from typing import TypedDict, Annotated, Literal
import operator

model = ChatOpenAI(model="gpt-4.1")

@tool
def search(query: str) -> str:
    """정보를 검색합니다."""
    return f"'{query}' 검색 결과: LangGraph는 상태 기반 에이전트를 구축하는 프레임워크입니다."

tools = [search]
tools_by_name = {t.name: t for t in tools}
model_with_tools = model.bind_tools(tools)

class AgentState(TypedDict):
    messages: Annotated[list[AnyMessage], operator.add]

def llm_node(state: AgentState) -> dict:
    return {"messages": [model_with_tools.invoke(
        [SystemMessage(content="당신은 유용한 어시스턴트입니다.")] + state["messages"],
        config=lf_config,
)]}

def tool_node(state: AgentState) -> dict:
    results = []
    for tc in state["messages"][-1].tool_calls:
        result = tools_by_name[tc["name"]].invoke(tc["args"], config=lf_config)
        results.append(ToolMessage(content=str(result), tool_call_id=tc["id"]))
    return {"messages": results}

def should_continue(state: AgentState) -> Literal["tools", "__end__"]:
    return "tools" if state["messages"][-1].tool_calls else END

builder = StateGraph(AgentState)
builder.add_node("llm", llm_node)
builder.add_node("tools", tool_node)
builder.add_edge(START, "llm")
builder.add_conditional_edges("llm", should_continue, ["tools", END])
builder.add_edge("tools", "llm")
agent = builder.compile(checkpointer=InMemorySaver())

print("스트리밍 데모를 위한 에이전트 준비 완료")

스트리밍 데모를 위한 에이전트 준비 완료


In [2]:
# Observability 설정 (선택) - LangSmith 또는 Langfuse
# .env에 키를 설정하거나, 아래 주석을 해제하여 직접 입력하세요.
# os.environ["LANGFUSE_SECRET_KEY"] = "sk-lf-..."
# os.environ["LANGFUSE_PUBLIC_KEY"] = "pk-lf-..."
# os.environ["LANGFUSE_HOST"] = "https://lf.ddok.ai"
import os

# LangSmith: LANGSMITH_TRACING=true 시 자동 활성화 (코드 수정 불필요)
if os.environ.get("LANGSMITH_TRACING", "").lower() == "true":
    os.environ.setdefault("LANGCHAIN_TRACING_V2", "true")
    os.environ.setdefault("LANGCHAIN_API_KEY", os.environ.get("LANGSMITH_API_KEY", ""))
    os.environ.setdefault("LANGCHAIN_PROJECT", os.environ.get("LANGSMITH_PROJECT", "default"))
    print(f"LangSmith tracing ON — project: {os.environ['LANGCHAIN_PROJECT']}")

# Langfuse: invoke/stream 호출 시 config={"callbacks": [langfuse_handler]} 전달
langfuse_handler = None
if os.environ.get("LANGFUSE_SECRET_KEY"):
    from langfuse.langchain import CallbackHandler
    langfuse_handler = CallbackHandler()
    print(f"Langfuse tracing ON — {os.environ.get('LANGFUSE_HOST', '')}")
# Langfuse config: pass to invoke/stream/batch calls
lf_config = {"callbacks": [langfuse_handler]} if langfuse_handler else {}


Langfuse tracing ON — https://lf.ddok.ai


## 7.2 스트리밍 모드 비교

LangGraph는 다양한 스트리밍 모드를 제공합니다. 각 모드는 서로 다른 수준의 정보를 실시간으로 전달합니다.

| 모드 | 설명 | 용도 |
|------|------|------|
| `values` | 각 단계의 전체 상태 | 디버깅, 상태 추적 |
| `updates` | 각 노드가 반환한 업데이트만 | 진행 상황 표시 |
| `messages` | 메시지 토큰 단위 | 채팅 UI |
| `custom` | 사용자 정의 이벤트 | 커스텀 진행률 |
| `debug` | 전체 디버그 정보 | 개발 중 디버깅 |

## 7.3 stream_mode="values" — 전체 상태 스냅샷

`values` 모드는 각 노드 실행 후 **전체 상태**를 반환합니다. 그래프가 어떻게 진행되는지 전체적인 흐름을 추적할 때 유용합니다.

In [3]:
config = {"configurable": {"thread_id": "stream-values"}}
print("=== stream_mode='values' ===")
for state in agent.stream(
    {"messages": [HumanMessage(content="LangGraph 정보를 검색해 주세요")]},
    {**config, **lf_config}, stream_mode="values",
):
    msg = state["messages"][-1]
    print(f"  [{type(msg).__name__}] {str(msg.content)[:100]}")

=== stream_mode='values' ===
  [HumanMessage] LangGraph 정보를 검색해 주세요


  [AIMessage] 
  [ToolMessage] 'LangGraph 정보' 검색 결과: LangGraph는 상태 기반 에이전트를 구축하는 프레임워크입니다.


  [AIMessage] LangGraph는 상태 기반 에이전트를 구축하는 프레임워크입니다. 이를 통해 개발자는 복잡한 상태 관리가 필요한 에이전트를 효과적으로 설계하고 구현할 수 있습니다. 추가적인 정보


## 7.4 stream_mode="updates" — 노드별 업데이트

`updates` 모드는 각 노드가 **반환한 업데이트 값만** 전달합니다. 어떤 노드가 어떤 변경을 만들었는지 정확히 파악할 수 있습니다.

In [4]:
config = {"configurable": {"thread_id": "stream-updates"}}
print("=== stream_mode='updates' ===")
for chunk in agent.stream(
    {"messages": [HumanMessage(content="LangGraph 정보를 검색해 주세요")]},
    {**config, **lf_config}, stream_mode="updates",
):
    for node_name, update in chunk.items():
        print(f"\n  [{node_name}]")
        for msg in update.get("messages", []):
            if hasattr(msg, 'tool_calls') and msg.tool_calls:
                for tc in msg.tool_calls:
                    print(f"    Tool: {tc['name']}({tc['args']})")
            elif msg.content:
                print(f"    {msg.content[:150]}")

=== stream_mode='updates' ===



  [llm]
    Tool: search({'query': 'LangGraph 정보'})

  [tools]
    'LangGraph 정보' 검색 결과: LangGraph는 상태 기반 에이전트를 구축하는 프레임워크입니다.



  [llm]
    LangGraph는 상태 기반 에이전트를 구축하는 프레임워크입니다. 즉, 다양한 상태와 조건에 따라 동작하는 에이전트를 효과적으로 개발할 수 있게 해주는 도구입니다. 만약 더 자세한 개념, 구조, 활용 예시 등이 필요하다면 추가로 설명드릴 수 있습니다. 추가 정보가 필


## 7.5 stream_mode="messages" — 토큰 단위 스트리밍

`messages` 모드는 LLM이 생성하는 **토큰을 실시간으로** 전달합니다. 채팅 UI에서 타이핑 효과를 구현할 때 가장 적합합니다.

In [5]:
config = {"configurable": {"thread_id": "stream-messages"}}
print("=== stream_mode='messages' ===")
print("응답: ", end="")
for msg, metadata in agent.stream(
    {"messages": [HumanMessage(content="LangGraph란 무엇인가요? 간단히 답해 주세요.")]},
    {**config, **lf_config}, stream_mode="messages",
):
    if msg.content and metadata.get("langgraph_node") == "llm":
        print(msg.content, end="", flush=True)
print()

=== stream_mode='messages' ===
응답: 

Lang

Graph

는

 L

...

 도

와

줍

니다

.

## 7.6 여러 스트리밍 모드 동시 사용

`stream_mode`에 리스트를 전달하면 여러 모드를 **동시에** 사용할 수 있습니다. 반환되는 이벤트는 `(mode, data)` 튜플 형태입니다.

In [6]:
config = {"configurable": {"thread_id": "stream-multi"}}
print("=== 여러 스트리밍 모드 동시 사용 ===")
for event in agent.stream(
    {"messages": [HumanMessage(content="Python을 검색해 주세요")]},
    {**config, **lf_config}, stream_mode=["updates", "messages"],
):
    kind = event[0] if isinstance(event, tuple) else "unknown"
    print(f"  [{kind}] 수신됨")

=== 여러 스트리밍 모드 동시 사용 ===


  [messages] 수신됨
  [messages] 수신됨
  [messages] 수신됨
  [messages] 수신됨
  [messages] 수신됨
  [messages] 수신됨
  [messages] 수신됨
  [messages] 수신됨
  [messages] 수신됨
  [updates] 수신됨
  [messages] 수신됨
  [updates] 수신됨


  [messages] 수신됨
  [messages] 수신됨
  [messages] 수신됨
  [messages] 수신됨
  [messages] 수신됨
  [messages] 수신됨
  [messages] 수신됨
  [messages] 수신됨
  [messages] 수신됨
  [messages] 수신됨
  [messages] 수신됨
  [messages] 수신됨
  [messages] 수신됨


  [messages] 수신됨
  [messages] 수신됨
  [messages] 수신됨
  [messages] 수신됨
  [messages] 수신됨
  [messages] 수신됨
  [messages] 수신됨
  [messages] 수신됨
  [messages] 수신됨
  [messages] 수신됨
  [messages] 수신됨
  [messages] 수신됨
  [messages] 수신됨
  [messages] 수신됨
  [messages] 수신됨
  [messages] 수신됨
  [messages] 수신됨
  [messages] 수신됨
  [messages] 수신됨
  [messages] 수신됨


  [messages] 수신됨
  [messages] 수신됨
  [messages] 수신됨
  [messages] 수신됨
  [messages] 수신됨
  [messages] 수신됨
  [messages] 수신됨
  [messages] 수신됨
  [messages] 수신됨
  [messages] 수신됨


  [messages] 수신됨
  [messages] 수신됨
  [messages] 수신됨
  [messages] 수신됨
  [messages] 수신됨
  [messages] 수신됨
  [messages] 수신됨
  [messages] 수신됨
  [messages] 수신됨
  [messages] 수신됨
  [messages] 수신됨
  [messages] 수신됨


  [messages] 수신됨
  [messages] 수신됨
  [messages] 수신됨
  [messages] 수신됨
  [messages] 수신됨
  [messages] 수신됨
  [messages] 수신됨
  [messages] 수신됨
  [messages] 수신됨
  [messages] 수신됨
  [messages] 수신됨
  [messages] 수신됨
  [messages] 수신됨
  [updates] 수신됨


## 7.7 stream_mode="custom" — 사용자 정의 스트리밍

`custom` 모드는 노드 내부에서 **임의의 데이터를 직접 스트리밍**할 수 있게 해줍니다.

`langgraph.config`의 `get_stream_writer()`를 호출하면 `writer` 함수를 얻을 수 있고, 이 함수에 딕셔너리 등의 데이터를 전달하면 그래프 실행 중 실시간으로 클라이언트에 전송됩니다.

**활용 사례:**
- 긴 작업의 **진행률(progress)** 보고
- LangChain을 사용하지 않는 외부 LLM의 **청크 단위 스트리밍**
- 노드 내부의 **중간 결과**를 즉시 전달

> `stream_mode="custom"`으로 그래프를 스트리밍하면 `writer()`로 전송한 데이터만 수신됩니다.

In [7]:
from langgraph.config import get_stream_writer

# --- 커스텀 스트리밍을 사용하는 간단한 그래프 정의 ---
class CustomState(TypedDict):
    topic: str
    result: str

def research_node(state: CustomState) -> dict:
    """research_node: get_stream_writer()로 중간 진행 상황을 스트리밍합니다."""
    writer = get_stream_writer()

    # 1단계: 검색 시작 알림
    writer({"step": 1, "status": "검색 중", "detail": f"'{state['topic']}' 검색 중 ..."})

    # 2단계: 분석 중 알림
    writer({"step": 2, "status": "분석 중", "detail": "검색 결과 분석 중 ..."})

    # 3단계: 완료 알림
    writer({"step": 3, "status": "완료", "detail": "조사 완료."})

    return {"result": f"'{state['topic']}'에 대한 조사가 완료되었습니다."}

custom_builder = StateGraph(CustomState)
custom_builder.add_node("research", research_node)
custom_builder.add_edge(START, "research")
custom_builder.add_edge("research", END)
custom_graph = custom_builder.compile()

# --- stream_mode="custom" 으로 실행 ---
print("=== stream_mode='custom' ===")
for chunk in custom_graph.stream(
    {"topic": "LangGraph 스트리밍"},
    stream_mode="custom",
    config=lf_config,
):
    print(f"  커스텀 이벤트: {chunk}")

print("\n--- updates + custom 동시 사용 ---")
for mode, chunk in custom_graph.stream(
    {"topic": "LangGraph 스트리밍"},
    stream_mode=["updates", "custom"],
    config=lf_config,
):
    if mode == "custom":
        print(f"  [custom]  {chunk}")
    else:
        print(f"  [updates] {chunk}")

=== stream_mode='custom' ===
  커스텀 이벤트: {'step': 1, 'status': '검색 중', 'detail': "'LangGraph 스트리밍' 검색 중 ..."}
  커스텀 이벤트: {'step': 2, 'status': '분석 중', 'detail': '검색 결과 분석 중 ...'}
  커스텀 이벤트: {'step': 3, 'status': '완료', 'detail': '조사 완료.'}

--- updates + custom 동시 사용 ---
  [custom]  {'step': 1, 'status': '검색 중', 'detail': "'LangGraph 스트리밍' 검색 중 ..."}
  [custom]  {'step': 2, 'status': '분석 중', 'detail': '검색 결과 분석 중 ...'}
  [custom]  {'step': 3, 'status': '완료', 'detail': '조사 완료.'}
  [updates] {'research': {'result': "'LangGraph 스트리밍'에 대한 조사가 완료되었습니다."}}


## 7.8 version="v2" — 타입-안전 통일 스트림 (LangGraph 1.1+)

`version="v2"`를 opt-in하면 **모든 청크가 `StreamPart` dict**로 통일된다. v1은 모드/서브그래프 조합에 따라 dict/tuple이 섞여 호출 측 분기 코드가 복잡했다.

### StreamPart 구조

```python
{
    "type": "values" | "updates" | "messages" | "custom" | ...,
    "ns":   (),    # 서브그래프 namespace 튜플
    "data": ...,   # 모드별 payload
}
```

### 이점

- **타입 안전성**: `langgraph.types`의 `UpdatesStreamPart`, `CustomStreamPart` 등으로 편집기가 `data`를 자동 내로잉
- **일관된 분기 코드**: 항상 `chunk["type"]`으로 분기 → v1처럼 tuple vs dict 판별 불필요
- **Pydantic / dataclass 자동 강제**: `stream_mode="values"`에서 state가 선언된 타입으로 coerce
- **후방 호환**: v1 동작 유지, v2는 순수 opt-in

In [ ]:
# 앞서 만든 agent에 version="v2"를 그대로 적용
config = {"configurable": {"thread_id": "stream-v2"}}
print("=== version='v2' + multi-mode ===")
for chunk in agent.stream(
    {"messages": [HumanMessage(content="LangGraph 정보를 검색해 주세요")]},
    {**config, **lf_config},
    stream_mode=["updates", "messages"],
    version="v2",
):
    # v1에서는 (mode, payload) tuple — v2는 항상 StreamPart dict
    kind = chunk["type"]
    ns   = chunk["ns"]
    data = chunk["data"]

    if kind == "updates":
        for node, upd in data.items():
            msgs = upd.get("messages", [])
            if msgs:
                m = msgs[-1]
                tag = "tool_call" if getattr(m, 'tool_calls', None) else type(m).__name__
                print(f"  [updates/{node}] {tag}")
    elif kind == "messages":
        msg, meta = data
        if msg.content and meta.get("langgraph_node") == "llm":
            # 토큰 단위 출력
            print(msg.content, end="", flush=True)
print()

### v1 → v2 비교

| 측면 | v1 (기본) | v2 (opt-in) |
|------|-----------|-------------|
| 반환 형태 | 모드/서브그래프 조합에 따라 dict/tuple 혼재 | 항상 `StreamPart` dict |
| 모드 식별 | tuple 첫 원소(다중모드) / 암묵적 | `chunk["type"]` 명시 필드 |
| namespace | `subgraphs=True` 시에만 tuple 첫 원소 | 항상 `chunk["ns"]` |
| 타입 추론 | 제한적 | TypedDict로 자동 내로잉 |
| Pydantic/dataclass state | 유지 (dict) | values 모드에서 자동 인스턴스화 |

### 마이그레이션 전략

1. 새 코드는 **`version="v2"` 기본값**으로 작성
2. 기존 v1 호출부는 **그대로 두어도 무방** (v1은 계속 동작)
3. 타입 안전성이 필요한 핫패스 / 신기능부터 점진 마이그레이션
4. Pydantic state 쓰는 그래프는 자동 강제 덕분에 이득이 가장 큼

## 7.9 요약

| 스트리밍 모드 | 설명 | 용도 |
|-------------|------|------|
| `values` | 각 단계 후 전체 상태 반환 | 디버깅, 상태 추적 |
| `updates` | 노드가 변경한 부분만 반환 | 진행 상황 모니터링 |
| `messages` | LLM 토큰 실시간 스트리밍 | 채팅 UI 구현 |
| 여러 모드 동시 | 리스트로 전달 → `(mode, data)` 튜플 수신 | 복합 모니터링 |
| `custom` | `get_stream_writer()`로 임의 데이터 전송 | 진행률 보고, 외부 LLM |

### 다음 단계
→ **[08_interrupts_and_time_travel.ipynb](./08_interrupts_and_time_travel.ipynb)**: 인터럽트와 타임 트래블을 배웁니다.